In [11]:
# Importation des modules
# Import bibliothèque de manipulation de dataframe
import pandas as pd

# Import des bibliothèques de viz
import matplotlib.pyplot as plt
import seaborn as sns

# Import split data
from sklearn.model_selection import train_test_split

# Import modèles de ML Supervisé Régression
from sklearn.linear_model import LinearRegression

# Import modèles de ML Supervisé Classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Import modèle de ML NON Supervisé
from sklearn.neighbors import NearestNeighbors

# Import des métriques
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Import outil standardisation de la donnée
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer

# Import pipeline
from sklearn.pipeline import Pipeline

from sklearn.base import BaseEstimator, TransformerMixin

# Gestion des warnings
import warnings

import ast

In [12]:
# Custom transformer for MultiLabelBinarizer
class MultiLabelBinarizerPipelineFriendly(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()

    def fit(self, X, y=None):
        self.mlb.fit(X)
        return self

    def transform(self, X):
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_

In [13]:
# Récuperation du df
df = pd.read_csv('../ressources/df_ml.csv', sep=';', encoding='utf-8')
df.isna().sum()

frenchTitle      0
genres           0
averageRating    0
numVotes         0
directors        0
decade           0
actors           0
dtype: int64

----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Preprocessor
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
X = df.drop(columns=['frenchTitle'])
films_non_standardise = X.iloc[:3]

In [15]:
# Fonction pour alourdir la valeur des colonnes
def multiply_block(X, factor):
    return X * factor

In [16]:
# Preprocessor pour standardiser les colonnes numériques
preprocessor = ColumnTransformer(
    transformers=[
        ('actors', Pipeline([
            ('mlb', MultiLabelBinarizerPipelineFriendly()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 2))),
            ]), 'actors'),
        ('directors', Pipeline([
            ('mlb', MultiLabelBinarizerPipelineFriendly()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 0.8))),
            ]), 'directors'),
        ('genres', MultiLabelBinarizerPipelineFriendly(), 'genres'),
        ('decade', Pipeline([
            ('encoder', OrdinalEncoder()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 0.5))),
            ]), ['decade']),
        ('scaler', StandardScaler(), ['averageRating']),
        ('numVotes', Pipeline([
            ('scaler', StandardScaler()),
            ('weight', FunctionTransformer(lambda x: multiply_block(x, 0.2))),
            ]), ['numVotes'])
    ]
)


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Pipeline
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
# Création du pipeline
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('knn', NearestNeighbors(n_neighbors=11))
    ]
)

pipeline.fit(X)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('actors',
                                                  Pipeline(steps=[('mlb',
                                                                   MultiLabelBinarizerPipelineFriendly()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda> at 0x000001C9FC49FBA0>))]),
                                                  'actors'),
                                                 ('directors',
                                                  Pipeline(steps=[('mlb',
                                                                   MultiLabelBinarizerPipelineFriendly()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda...
                                                  Pipeline(steps=[('encoder',
                                                                   OrdinalEncoder()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda> at 0x000001C9D29AE480>))]),
                                                  ['decade']),
                                                 ('scaler', StandardScaler(),
                                                  ['averageRating']),
                                                 ('numVotes',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler()),
                                                                  ('weight',
                                                                   FunctionTransformer(func=<function <lambda> at 0x000001C9D29AED40>))]),
                                                  ['numVotes'])])),
                ('knn', NearestNeighbors(n_neighbors=11))])

In [18]:
point_test = X.iloc[:3]
# Prédiction des voisins les plus proches
X_test_transformed = pipeline.named_steps['preprocessor'].transform(point_test)

distances, indices = pipeline.named_steps['knn'].kneighbors(X_test_transformed)
# Affichage des indices des voisins les plus proches
print("Indices des voisins les plus proches :", indices)
# Affichage des distances des voisins les plus proches
print("Distances des voisins les plus proches :", distances)


Indices des voisins les plus proches : [[   0   37   73   36   74  538 1503   39 1311  248  556]
 [   1    6  463  372   66  493 8557    9  293 1641  719]
 [   2  960  874 1590  583 1134 1839  295  849   67  844]]
Distances des voisins les plus proches : [[0.         5.66904995 6.26743431 6.46269554 6.58882875 6.61027547
  6.6363993  6.72445279 6.7291087  6.741332   6.84702123]
 [0.         5.59087573 5.73359842 5.92769294 6.04732437 6.1885336
  6.25614324 6.42323135 6.47693599 6.52687053 6.5269716 ]
 [0.         7.19527816 7.43985073 7.49907509 7.50569339 7.53990336
  7.68843271 7.77890485 7.82610129 7.88782898 7.90710007]]


In [26]:
titres = ['Karaté Kid', "Transformer", 'Avatar']  # ou d’autres

for titre in titres:
    film_cible = df[df['frenchTitle'].str.contains(titre, case=False, na=False)]
    if film_cible.empty:
        print(f"Film '{titre}' non trouvé.")
        continue

    idx_film = film_cible.index[0]
    film_non_standardise = df.drop(columns=['frenchTitle']).loc[[idx_film]]
    film_transforme = pipeline.named_steps['preprocessor'].transform(film_non_standardise)
    distances, indices = pipeline.named_steps['knn'].kneighbors(film_transforme)

    print(f"\n🎬 Film : {df.loc[idx_film, 'frenchTitle']} (Index: {idx_film})")
    print(f"  Note moyenne : {df.loc[idx_film, 'averageRating']}")
    neighbor_original_indices = X.iloc[indices[0]].index
    neighbor_info = df.loc[neighbor_original_indices][['frenchTitle', 'averageRating', 'numVotes', 'decade', 'genres', 'actors', 'directors']]
    print("  Voisins :")
    display(neighbor_info)


🎬 Film : Miss Karaté Kid (Index: 3448)
  Note moyenne : 4.6
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actors,directors
3448,Miss Karaté Kid,4.6,36125,1990,"Drama, Family, Adventure, Action","['Pat Morita', 'Hilary Swank', 'Michael Ironsi...",Christopher Cain
3521,Le cygne et la princesse,6.4,28219,1990,"Adventure, Comedy, Family, Fantasy, Animation","['Jack Palance', 'Howard McGillin', 'Michelle ...",Richard Rich
16269,Midway,6.7,100782,2010,"Adventure, History, War, Drama, Action","['Ed Skrein', 'Patrick Wilson', 'Woody Harrels...",Roland Emmerich
3105,Un père en cavale,5.0,3625,1990,"Adventure, Crime, Family, Comedy, Action","['Patrick Swayze', 'Halle Berry', 'Sabrina Llo...",Darrell Roodt
4204,À couteaux tirés,7.0,82491,1990,"Drama, Adventure, Action","['Anthony Hopkins', 'Alec Baldwin', 'Elle Macp...",Lee Tamahori
14716,Smosh: The Movie,3.3,6993,2010,"Adventure, Action, Comedy, Fantasy","['Ian Hecox', 'Anthony Padilla', 'Michael Ian ...",Alex Winter
10623,Sinbad: The Fifth Voyage,3.4,3178,2010,"Family, Adventure, Action, Fantasy","['Patrick Stewart', 'Shahin Sean Solimon', 'Sa...",Shahin Sean Solimon
14565,David and Goliath,4.3,2548,2010,"Drama, Adventure, Action, Fantasy","['Miles Sloman', 'Jerald Sokolowski', 'Paul Hu...",Timothy A. Chey
9824,One Night in Bangkok,5.2,2974,2020,"Crime, Drama, Action, Thriller","['Mark Dacascos', 'Vanida Golten', 'Prinya Int...",Wych Kaosayananda
4773,High Art,6.6,11383,1990,"Drama, Romance","['Radha Mitchell', 'Ally Sheedy', 'Patricia Cl...",Lisa Cholodenko



🎬 Film : La Guerre des robots: Transformers (Index: 2236)
  Note moyenne : 7.2
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actors,directors
2236,La Guerre des robots: Transformers,7.2,43651,1980,"Adventure, Action, Family, Animation, ScienceF...","['Orson Welles', 'Robert Stack', 'Leonard Nimoy']",Nelson Shin
2831,Aladdin,8.0,487186,1990,"Adventure, Comedy, Family, Fantasy, Animation,...","['Scott Weinger', 'Robin Williams', 'Linda Lar...","Ron Clements, John Musker"
2909,"Chérie, j'ai agrandi le bébé",4.9,48267,1990,"Adventure, ScienceFiction, Comedy, Family","['Rick Moranis', 'Marcia Strassman', 'Robert O...",Randal Kleiser
16144,Carnage,7.6,1606,2010,"Comedy, ScienceFiction","['Racheal Ofori', 'Myles Sembi', 'Robert Wilde']",Simon Amstell
15970,Les traducteurs,6.5,10094,2010,"Mystery, Thriller","['Lambert Wilson', 'Olga Kurylenko', 'Riccardo...",Régis Roinsard
700,La dernière rafale,7.0,3493,1940,"Crime, Drama, Action, Thriller","['Mark Stevens', 'Richard Widmark', 'Lloyd Nol...",William Keighley
15232,"Everything, Everything",6.3,43775,2010,"Drama, Romance","['Amandla Stenberg', 'Nick Robinson', 'Anika N...",Stella Meghie
1405,La Grande Course autour du monde,7.2,20805,1960,"Adventure, Family, Comedy, Action, Romance","['Tony Curtis', 'Natalie Wood', 'Jack Lemmon']",Blake Edwards
13532,La Nuit au musée : Le Secret des pharaons,6.2,143054,2010,"Fantasy, Adventure, Comedy, Family","['Ben Stiller', 'Robin Williams', 'Owen Wilson']",Shawn Levy
3451,Absolom 2022,6.1,23619,1990,"Sci-Fi, Action, Drama, ScienceFiction, Thriller","['Ray Liotta', 'Lance Henriksen', 'Stuart Wils...",Martin Campbell



🎬 Film : Avatar (Index: 7976)
  Note moyenne : 7.9
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actors,directors
7976,Avatar,7.9,1432784,2000,"ScienceFiction, Adventure, Action, Fantasy","['Sam Worthington', 'Zoe Saldaña', 'Sigourney ...",James Cameron
11488,Avatar : La Voie de l'eau,7.5,535737,2020,"ScienceFiction, Adventure, Action, Fantasy","['Sam Worthington', 'Zoe Saldaña', 'Sigourney ...",James Cameron
11304,Mortal Engines,6.1,148531,2010,"ScienceFiction, Adventure, Action, Fantasy","['Hera Hilmar', 'Robert Sheehan', 'Hugo Weaving']",Christian Rivers
6405,"I, Robot",7.1,597510,2000,"ScienceFiction, Mystery, Action, Sci-Fi","['Will Smith', 'Bridget Moynahan', 'Bruce Gree...",Alex Proyas
14172,La planète des singes : Suprématie,7.4,316742,2010,"Adventure, Action, War, Drama, ScienceFiction","['Andy Serkis', 'Woody Harrelson', 'Steve Zahn']",Matt Reeves
10642,Star Trek Into Darkness,7.7,504320,2010,"ScienceFiction, Adventure, Action, Sci-Fi","['Chris Pine', 'Zachary Quinto', 'Zoe Saldaña']",J.J. Abrams
9826,Le royaume de Ga'Hoole - La légende des gardiens,6.9,89828,2010,"Adventure, Family, Fantasy, Animation, Action","['Jim Sturgess', 'Hugo Weaving', 'David Wenham']",Zack Snyder
4632,Small Soldiers,6.3,108743,1990,"Adventure, Action, Fantasy, Comedy, ScienceFic...","['Kirsten Dunst', 'Gregory Smith', 'David Cross']",Joe Dante
17406,Messagères de guerre,7.6,521,2020,"Drame, Guerre, Histoire","['Kerry Washington', 'Sam Waterston', 'Susan S...",Tyler Perry
14534,Les Gardiens de la Galaxie Vol. 2,7.6,796731,2010,"ScienceFiction, Adventure, Action, Comedy","['Chris Pratt', 'Zoe Saldaña', 'Dave Bautista']",James Gunn
